# Stage 5 -- Interpretability and label robustness (E5, E6)
(1) Compares the LGC co-occurrence adjacency and the LAT label-label attention against
the empirical co-occurrence matrix (Spearman correlation + heatmaps).
(2) Label-noise robustness using the per-tool predictions: evaluation on the uncontested
subset and re-training on majority-vote labels.
**Accelerator: GPU.** **Attach:** `scvd-lib`, `dive-smart-contracts`, `scvd-stage1-eda`,
`scvd-stage3-deep` (test predictions).

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd
LIB = '/kaggle/input/scvd-lib'
if not os.path.isdir(LIB):
    LIB = os.environ.get('SCVD_LIB', os.path.abspath('..'))   # local fallback
sys.path.insert(0, LIB)
import scvd
root, log = scvd.setup_stage('stage5_interpretability')
SEEDS = (0,) if scvd.quick() else scvd.RNG_SEEDS
log.info('seeds=%s quick=%s', SEEDS, scvd.quick())

In [ ]:
pre, post, lab, tools = scvd.load_dive(log)
fam = scvd.family_columns(pre, post)
for k, v in fam.items():
    log.info('family %-18s %3d features', k, len(v))
for v in scvd.VIEWS:
    log.info('view F_%-4s %3d features', v, len(scvd.view_columns(fam, v)))

In [ ]:
splits, sdf = scvd.load_splits() or (scvd.build_splits(pre, lab, 0), None)
d = scvd.make_data(pre, post, lab, fam, 'all', *splits['S1'])
M = scvd.cooccurrence(d['y_tr'])
cfg = {'lr': 1e-3, 'dm': 128, 'layers': 3, 'tau': 0.2, 'batch': 256,
       'epochs': 60 if scvd.quick() else 120}
lgc = scvd.LGC(d['fams'], M, dm=128, tau=0.2)
_, _h = scvd.train_torch(lgc, d, cfg, log)
A = scvd.lgc_A(M, 0.2)
lat = scvd.LAT(d['fams'], dm=128, layers=3)
_, _h = scvd.train_torch(lat, d, cfg, log)
attn = scvd.extract_label_attention(lat, d)
assert not np.allclose(attn, np.eye(8)), 'attention extraction returned identity fallback'
from scipy.stats import spearmanr
off = ~np.eye(8, dtype=bool)
rho_lgc = spearmanr(M[off], A[off]).statistic
rho_lat = spearmanr(M[off], attn[off]).statistic
pd.DataFrame({'pair': [f'{scvd.LABEL_SHORT[i]}-{scvd.LABEL_SHORT[j]}'
                       for i in range(8) for j in range(8) if off[i, j]],
              'M': M[off], 'A_lgc': A[off], 'attn_lat': attn[off]}).to_csv(
    f'{root}/results/stage5_dependency_pairs.csv', index=False)
log.info('Spearman(M, A_lgc)=%.3f  Spearman(M, attn_lat)=%.3f', rho_lgc, rho_lat)
scvd.plot_dependency_maps(M, A, attn, root, log)

In [ ]:
Y = lab.set_index('contractID')[scvd.LABELS]
tl = tools.set_index('contractID')
if tl.index.duplicated().any():
    tl = tl[~tl.index.duplicated(keep='first')]
tl = tl.reindex(Y.index)   # Tool_Results covers more contracts than the label file
tool_cols = {j: [c for c in tl.columns if c.endswith(f'_{j+1}')] for j in range(8)}
agree = pd.DataFrame(True, index=Y.index, columns=scvd.LABELS)
for j, col in enumerate(scvd.LABELS):
    tc = tl[tool_cols[j]]
    obs = tc.notna()
    vals = tc.fillna(-1).eq(Y[col].values[:, None] * 1.0).where(obs, True)
    agree[col] = vals.all(1)
uncontested = agree.all(1)
log.info('uncontested contracts: %d (%.1f%%)', uncontested.sum(), 100 * uncontested.mean())
maj = pd.DataFrame(np.nan, index=Y.index, columns=scvd.LABELS)
for j, col in enumerate(scvd.LABELS):
    tc = tl[tool_cols[j]]
    s = tc.sum(1); n = tc.notna().sum(1)
    maj[col] = np.where(n > 0, (s / n.clip(lower=1) > 0.5).astype(float), np.nan)
maj = maj.fillna(Y.astype(float))
res = {}
ids_te = d['ids_te']
for name, mdl_factory in [('LGC', lambda: scvd.LGC(d['fams'], M, dm=128, tau=0.2)),
                          ('LAT', lambda: scvd.LAT(d['fams'], dm=128, layers=3)),
                          ('BR-trunk', lambda: scvd.LAT(d['fams'], dm=128, layers=3, trunk_only=True))]:
    mdl = mdl_factory(); pt, _h = scvd.train_torch(mdl, d, cfg, log)
    th = scvd.tune_thresholds(d['y_va'], scvd.predict(mdl, d, 'va'))
    pred = (pt >= th).astype(int)
    yte = d['y_te']
    full = scvd.all_metrics(yte, pred, pt)['macro_f1']
    mask = uncontested.reindex(ids_te).to_numpy()
    ag = scvd.all_metrics(yte[mask], pred[mask], pt[mask])['macro_f1']
    ymaj = maj.reindex(ids_te)[scvd.LABELS].to_numpy(int)
    d2 = dict(d); d2['y_tr'] = maj.reindex(d['ids_tr'])[scvd.LABELS].to_numpy(int)
    d2['y_va'] = maj.reindex(d['ids_va'])[scvd.LABELS].to_numpy(int)
    d2['y_te'] = ymaj
    mdl2 = mdl_factory(); pt2, _h2 = scvd.train_torch(mdl2, d2, cfg, log)
    majo = scvd.all_metrics(ymaj, (pt2 >= 0.5).astype(int), pt2)['macro_f1']
    res[name] = {'full': full, 'agreement': ag, 'majority': majo}
    log.info('%-9s full=%.4f uncontested=%.4f majority-retrained=%.4f', name, full, ag, majo)
json.dump(res, open(f'{root}/results/stage5_robustness.json', 'w'), indent=1)
scvd.plot_robustness(res, root, log)

In [ ]:
url = scvd.push_dataset(root, 'stage5_interpretability', 'scvd-stage5-interpretability', 'SCVD Stage 5: interpretability and label-robustness analyses', log)
if url:
    print('DATASET:', url)
else:
    print('push skipped (no credentials); artifacts remain under', root)